# GridPulse Kaggle Forecast Training Template

Use this notebook to train the current `tree` forecaster on a Kaggle-uploaded canonical 15-minute AMI dataset.

Expected dataset files:

- `feeder.csv`
- `substation.csv`
- `fingerprints.csv`
- `run_config.json`


In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATASET_ROOT = Path('/kaggle/input/gridpulse-forecasting-inputs')
OUTPUT_ROOT = Path('/kaggle/working')
RUN_CONFIG = json.loads((DATASET_ROOT / 'run_config.json').read_text())
JOB = RUN_CONFIG['jobs'][0]
ENTITY_TYPE = JOB['entity_type']
ENTITY_ID = JOB['entity_id']
HORIZON = JOB['horizon']
DATASET_FILE = JOB['dataset_file']
HORIZON_TO_STEPS = {'1h': 4, '4h': 16, '24h': 96}
TARGET_FEATURE_COLUMNS = (
    'contracted_md_kw', 'temperature_c', 'humidity_percent', 'hour', 'minute',
    'slot_index', 'day_of_week', 'month', 'season_index', 'is_weekend',
    'is_holiday', 'production_schedule_kw', 'fingerprint_mean_kw',
    'fingerprint_p10_kw', 'fingerprint_p90_kw', 'lag_1', 'lag_4',
    'lag_16', 'lag_96', 'rolling_mean_4', 'rolling_mean_16', 'rolling_mean_96'
)


In [ ]:
history = pd.read_csv(DATASET_ROOT / DATASET_FILE)
fingerprints = pd.read_csv(DATASET_ROOT / 'fingerprints.csv')
history['timestamp'] = pd.to_datetime(history['timestamp'], utc=True)
history.head()


In [ ]:
def filter_target_history(frame: pd.DataFrame, entity_type: str, entity_id: str) -> pd.DataFrame:
    filtered = frame[
        (frame['entity_type'].astype(str).str.lower() == entity_type.lower())
        & (frame['entity_id'].astype(str) == entity_id)
    ].copy()
    if filtered.empty:
        raise ValueError(f'Unknown target: {entity_type} {entity_id}')
    return filtered.sort_values('timestamp').reset_index(drop=True)

def build_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    working = frame.copy()
    shifted_load = working['load_kw'].shift(1)
    working['lag_1'] = working['load_kw'].shift(1)
    working['lag_4'] = working['load_kw'].shift(4)
    working['lag_16'] = working['load_kw'].shift(16)
    working['lag_96'] = working['load_kw'].shift(96)
    working['rolling_mean_4'] = shifted_load.rolling(4, min_periods=1).mean()
    working['rolling_mean_16'] = shifted_load.rolling(16, min_periods=1).mean()
    working['rolling_mean_96'] = shifted_load.rolling(96, min_periods=1).mean()
    return working

def build_training_dataset(feature_frame: pd.DataFrame, horizon_steps: int) -> pd.DataFrame:
    rows = []
    base = feature_frame.dropna(subset=['lag_1', 'lag_4', 'lag_16', 'lag_96']).reset_index(drop=True)
    for index in range(len(base) - horizon_steps):
        target_sequence = base['load_kw'].iloc[index + 1:index + 1 + horizon_steps].astype(float).tolist()
        row = base.iloc[index].to_dict()
        row['target_sequence'] = target_sequence
        rows.append(row)
    return pd.DataFrame(rows)

def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    denominator = np.where(np.abs(y_true) < 1e-9, 1e-9, np.abs(y_true))
    mape = float(np.mean(np.abs((y_true - y_pred) / denominator)) * 100.0)
    r2 = float(r2_score(y_true, y_pred))
    true_peak_index = int(np.argmax(y_true))
    pred_peak_index = int(np.argmax(y_pred))
    peak_time_error = abs(pred_peak_index - true_peak_index)
    peak_load_error = float(abs(float(y_pred[pred_peak_index]) - float(y_true[true_peak_index])))
    return {
        'mae': round(mae, 6),
        'rmse': round(rmse, 6),
        'mape': round(mape, 6),
        'r2': round(r2, 6),
        'peak_time_error': peak_time_error,
        'peak_load_error': round(peak_load_error, 6),
    }


In [ ]:
horizon_steps = HORIZON_TO_STEPS[HORIZON]
target_history = filter_target_history(history, ENTITY_TYPE, ENTITY_ID)
feature_frame = build_feature_frame(target_history)
dataset = build_training_dataset(feature_frame, horizon_steps)
dataset[['timestamp', 'load_kw']].head()


In [ ]:
split_index = max(1, int(len(dataset) * 0.8))
if split_index >= len(dataset):
    split_index = len(dataset) - 1

train_frame = dataset.iloc[:split_index].copy()
test_frame = dataset.iloc[split_index:].copy()

x_train = train_frame.loc[:, list(TARGET_FEATURE_COLUMNS)]
y_train = np.vstack(train_frame['target_sequence'].to_list())
x_test = test_frame.loc[:, list(TARGET_FEATURE_COLUMNS)]
y_test = np.vstack(test_frame['target_sequence'].to_list())

model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=7,
    n_jobs=-1,
)
model.fit(x_train, y_train)
predictions = model.predict(x_test)
metrics = calculate_metrics(y_test.flatten(), predictions.flatten())
metrics


In [ ]:
residuals = y_test - predictions
artifact = {
    'model_name': 'tree',
    'entity_type': ENTITY_TYPE,
    'entity_id': ENTITY_ID,
    'horizon': HORIZON,
    'horizon_steps': horizon_steps,
    'feature_columns': list(TARGET_FEATURE_COLUMNS),
    'model': model,
    'metrics': metrics,
    'lower_residual': float(np.quantile(residuals, 0.10)),
    'upper_residual': float(np.quantile(residuals, 0.90)),
}
safe_entity_id = ENTITY_ID.replace('/', '_')
output_name = JOB.get('output_name', f'tree_{ENTITY_TYPE}_{safe_entity_id}_{HORIZON}')
artifact_path = OUTPUT_ROOT / f'{output_name}.pkl'
metrics_path = OUTPUT_ROOT / f'{output_name}.json'
with artifact_path.open('wb') as artifact_file:
    pickle.dump(artifact, artifact_file)
with metrics_path.open('w', encoding='utf-8') as metrics_file:
    json.dump({'job': JOB, 'metrics': metrics}, metrics_file, indent=2)
print(artifact_path)
print(metrics_path)
